# HotelIQ – Model Building and Evaluation

## Objective

The objective of this notebook is to develop machine learning models that predict hotel booking cancellations.

### Steps Performed

- Load the cleaned hotel booking dataset.
- Separate independent and dependent variables.
- Split the dataset into training and testing sets.
- Apply feature preprocessing.
- Train multiple classification algorithms.
- Evaluate model performance using classification metrics.
- Compare models and identify the best-performing model.

### Machine Learning Algorithms

1. Logistic Regression
2. Decision Tree Classifier
3. Random Forest Classifier
4. XGBoost Classifier

**Evaluation approach:** This notebook uses an estimated booking-date chronological split (70% train, 15% validation, 15% test). The booking date is approximated as arrival date minus lead time because an explicit booking-creation date is not available. `arrival_date_year` is excluded from model features; it is used only to construct the approximate chronology.

In [ ]:
# Importing required libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
df = pd.read_csv('../data/hotel_bookings_cleaned.csv')

df.shape

In [ ]:
df.head()

In [ ]:
df.info()

### Separating Features and Target

The target variable is `is_canceled`, which indicates whether a hotel booking was cancelled.

- **X:** Independent features used to predict booking cancellation.
- **y:** Target variable that the model will predict.

Columns that may cause data leakage or are unavailable at the time of booking are excluded.

In [ ]:
# Columns to exclude from model features
drop_columns = [
    'is_canceled',
    'arrival_date_year',
    'reservation_status',
    'reservation_status_date',
    'booking_changes',
    'assigned_room_type',
    'days_in_waiting_list'
]

# Separate features and target
X = df.drop(columns=drop_columns)
y = df['is_canceled']

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)

### Chronological Data Split

The dataset is ordered by an estimated booking date (arrival date minus lead time) and divided into 70% training, 15% validation and 15% testing data. The latest chronological test period remains untouched until final evaluation.

This approach better reflects a future-facing prediction task than random splitting. However, the estimated booking date is approximate and the source data covers only 2015–2017.

In [ ]:
# Estimate booking date (arrival date minus lead time)
month_numbers = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4,
    'May': 5, 'June': 6, 'July': 7, 'August': 8,
    'September': 9, 'October': 10, 'November': 11, 'December': 12
}

arrival_dates = pd.to_datetime({
    'year': df['arrival_date_year'],
    'month': df['arrival_date_month'].map(month_numbers),
    'day': df['arrival_date_day_of_month']
})
estimated_booking_dates = arrival_dates - pd.to_timedelta(df['lead_time'], unit='D')

# Sort rows by estimated booking date
order = estimated_booking_dates.sort_values(kind='stable').index
X_sorted = X.loc[order].reset_index(drop=True)
y_sorted = y.loc[order].reset_index(drop=True)
sorted_booking_dates = estimated_booking_dates.loc[order].reset_index(drop=True)

# Chronological split: 70% train, 15% validation, 15% test
train_end = int(len(X_sorted) * 0.70)
validation_end = int(len(X_sorted) * 0.85)

X_train = X_sorted.iloc[:train_end]
y_train = y_sorted.iloc[:train_end]
X_val = X_sorted.iloc[train_end:validation_end]
y_val = y_sorted.iloc[train_end:validation_end]
X_test = X_sorted.iloc[validation_end:]
y_test = y_sorted.iloc[validation_end:]

print("Training Features:", X_train.shape)
print("Validation Features:", X_val.shape)
print("Testing Features:", X_test.shape)

print("\nTraining period:", sorted_booking_dates.iloc[0], "to", sorted_booking_dates.iloc[train_end - 1])
print("Validation period:", sorted_booking_dates.iloc[train_end], "to", sorted_booking_dates.iloc[validation_end - 1])
print("Testing period:", sorted_booking_dates.iloc[validation_end], "to", sorted_booking_dates.iloc[-1])

### Identifying Numerical and Categorical Features

The independent features are divided into numerical and categorical groups.

- Numerical features will be standardized using StandardScaler.
- Categorical features will be encoded using OneHotEncoder.

The `agent` column is treated as categorical because it represents an agent identifier rather than a continuous numerical measurement.

In [ ]:
# Define categorical features
categorical_features = ['hotel', 'arrival_date_month', 'meal', 
                        'country', 'market_segment', 
                        'distribution_channel', 'reserved_room_type', 
                        'deposit_type', 'customer_type', 'agent']

# Define numerical features
numerical_features = [col for col in X.columns
                      if col not in categorical_features]

print("Numerical Features:", len(numerical_features))
print("Categorical Features:", len(categorical_features))

In [ ]:
# Create preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)])

print("Preprocessor created successfully!")

### 1. Logistic Regression

Logistic Regression is used as the baseline classification model for predicting hotel booking cancellations.

It estimates the probability of a booking cancellation based on the selected independent features.

The preprocessing transformer and Logistic Regression model are combined using a Scikit-learn Pipeline.

In [ ]:
# creating logistic regression pipeline

logistic_model = Pipeline(steps=[('preprocessor',preprocessor),
                          ('classifier',LogisticRegression(max_iter=1000, random_state=49))])

print("Logistics Regression pipeline created successfully!")


In [ ]:
#training model
logistic_model.fit(X_train, y_train)

print("Logistic Regression model trained successfully!")

### Logistic Regression Model Evaluation

The trained Logistic Regression model is evaluated using the testing dataset.

Predictions are generated to compare the model's predicted outcomes with the actual booking cancellation status.

In [ ]:
# Generate predictions
y_pred_logistic = logistic_model.predict(X_test)

# Generate prediction probabilities
y_prob_logistic = logistic_model.predict_proba(X_test)[:, 1]

print("Predictions generated successfully!")

print("\nFirst 10 Actual Values:")
print(y_test.head(10).values)

print("\nFirst 10 Predicted Values:")
print(y_pred_logistic[:10])

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix, 
                             classification_report)

#### Classification Performance Metrics

The Logistic Regression model is evaluated using the following metrics:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC score

In [ ]:
# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred_logistic)
precision = precision_score(y_test, y_pred_logistic)
recall = recall_score(y_test, y_pred_logistic)
f1 = f1_score(y_test, y_pred_logistic)
roc_auc = roc_auc_score(y_test, y_prob_logistic)

# Display results
print("Logistic Regression Performance")
print("--------------------------------")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")

#### Confusion Matrix

A confusion matrix helps us understand the number of correct and incorrect predictions made by the Logistic Regression model.

It shows:
- True Positives (TP)
- True Negatives (TN)
- False Positives (FP)
- False Negatives (FN)

In [ ]:
# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred_logistic)

print("Confusion Matrix:")
print(cm)

#### Classification Report

The classification report provides precision, recall, F1-score and support for both classes.

It helps evaluate how effectively the model predicts canceled and non-canceled bookings.

In [ ]:
print(classification_report(y_test, y_pred_logistic))

### 1. Logistic Regression – Overall Findings

The Logistic Regression model achieved an accuracy of 79.74% and an ROC-AUC score of 85.53%.

The model performs better at identifying non-canceled bookings than canceled bookings.

The recall for canceled bookings is 51.13%, indicating that the model misses a considerable number of actual cancellations.

Logistic Regression provides a useful baseline model. Other classification algorithms will be evaluated to determine whether cancellation prediction performance can be improved.

## 2. Decision Tree Classifier

A Decision Tree Classifier is a supervised machine learning algorithm that makes predictions using a tree-like structure of decision rules.

It splits the dataset based on feature values to classify hotel bookings as canceled or not canceled.

The model will use the same preprocessing steps and training dataset as Logistic Regression.

In [ ]:
# create decision tree pipeline

decision_tree_model = Pipeline(steps=[('preprocessor', preprocessor),
                                         ('classifier', DecisionTreeClassifier(random_state=49))])

#training model
decision_tree_model.fit(X_train, y_train)

print("Decision Tree model trained successfully!")

#### Decision Tree Predictions

The trained Decision Tree model is used to predict whether hotel bookings in the testing dataset will be canceled or not.

Prediction probabilities are also generated for calculating the ROC-AUC score.

In [ ]:
# Generate predictions
y_pred_dt = decision_tree_model.predict(X_test)

# Generate prediction probabilities
y_prob_dt = decision_tree_model.predict_proba(X_test)[:, 1]

print("Predictions generated successfully!")

print("\nFirst 10 Actual Values:")
print(y_test.head(10).values)

print("\nFirst 10 Predicted Values:")
print(y_pred_dt[:10])

#### Decision Tree – Performance Evaluation

The Decision Tree model is evaluated using accuracy, precision, recall, F1-score and ROC-AUC.

These metrics help measure how effectively the model identifies canceled and non-canceled hotel bookings.

In [ ]:
# Calculate evaluation metrics
accuracy_dt = accuracy_score(y_test, y_pred_dt)
precision_dt = precision_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)
roc_auc_dt = roc_auc_score(y_test, y_prob_dt)

# Display results
print("Decision Tree Performance")
print("--------------------------------")
print(f"Accuracy:  {accuracy_dt:.4f}")
print(f"Precision: {precision_dt:.4f}")
print(f"Recall:    {recall_dt:.4f}")
print(f"F1 Score:  {f1_dt:.4f}")
print(f"ROC-AUC:   {roc_auc_dt:.4f}")

In [ ]:
# Generate confusion matrix
cm_dt = confusion_matrix(y_test, y_pred_dt)

print("Decision Tree Confusion Matrix:")
print(cm_dt)

In [ ]:
#Decision Tree – Classification Report

#The classification report evaluates the Decision Tree model using precision, 
# recall, F1-score and support for both target classes.

print(classification_report(y_test, y_pred_dt))

### 2. Decision Tree – Overall Findings

The Decision Tree model achieved an accuracy of 80.39% and an ROC-AUC score of 75.59%.

Compared with Logistic Regression, the Decision Tree model achieved better recall and F1-score for canceled bookings.

It identified 64.29% of actual cancellations, compared with 51.13% for Logistic Regression.

However, its ROC-AUC score was lower, indicating weaker overall discrimination between canceled and non-canceled bookings.

The Decision Tree model provides a more balanced classification of the two classes, but further models will be evaluated to identify the best-performing approach.

## 3. Random Forest Classifier

Random Forest is an ensemble learning algorithm that combines multiple Decision Trees to make predictions.

Each tree is trained using a different sample of the training data, and their predictions are combined to produce the final classification.

Random Forest generally helps reduce overfitting compared with an individual Decision Tree.

In [ ]:
# create random forest pipeline

random_forest_model = Pipeline(steps=[('preprocessor', preprocessor),
                                      ('classifier', RandomForestClassifier(random_state=49, 
                                                                            n_estimators=100, n_jobs=-1))])

#train model
random_forest_model.fit(X_train, y_train)

print("Random Forest model trained successfully!")

#### Random Forest Predictions

The trained Random Forest model is used to predict whether hotel bookings in the testing dataset will be canceled or not.

Prediction probabilities are also generated to evaluate the model using the ROC-AUC score.

In [ ]:
# Generate predictions
y_pred_rf = random_forest_model.predict(X_test)

# Generate prediction probabilities
y_prob_rf = random_forest_model.predict_proba(X_test)[:, 1]

print("Predictions generated successfully!")

print("\nFirst 10 Actual Values:")
print(y_test.head(10).values)

print("\nFirst 10 Predicted Values:")
print(y_pred_rf[:10])

#### Random Forest – Performance Evaluation

The Random Forest model is evaluated using accuracy, precision, recall, F1-score and ROC-AUC.

These metrics help assess how effectively the model identifies canceled and non-canceled hotel bookings.

In [ ]:
# Calculate evaluation metrics
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)
roc_auc_rf = roc_auc_score(y_test, y_prob_rf)

# Display results
print("Random Forest Performance")
print("--------------------------------")
print(f"Accuracy:  {accuracy_rf:.4f}")
print(f"Precision: {precision_rf:.4f}")
print(f"Recall:    {recall_rf:.4f}")
print(f"F1 Score:  {f1_rf:.4f}")
print(f"ROC-AUC:   {roc_auc_rf:.4f}")

In [ ]:
# confusion matrix
cm_rf = confusion_matrix(y_test, y_pred_rf)

print("Random Forest Confusion Matrix:")
print(cm_rf)

In [ ]:
# classifcation report
print(classification_report(y_test, y_pred_rf))

### Random Forest – Overall Findings

This section reports Random Forest performance on the chronological test period. Run the notebook from top to bottom to refresh the results after the updated split and feature set. Do not reuse the previous random-split metrics, as they are not directly comparable.

## 4. XGBoost Classifier

XGBoost (Extreme Gradient Boosting) is an ensemble learning algorithm that builds multiple decision trees sequentially.

Each new tree tries to correct the errors made by the previous trees. This helps improve the model's predictive performance.

In [ ]:
from xgboost import XGBClassifier

In [ ]:
#The model will use 100 boosting rounds with a learning rate of 0.1 
#and a maximum tree depth of 6.

xgb_model = Pipeline(steps=[("preprocessor", preprocessor),
                            ('classifier', XGBClassifier(n_estimators=100,
                                                         learning_rate=0.1,
                                                         max_depth=6,
                                                         random_state=49,
                                                         eval_metric='logloss',
                                                         n_jobs=-1))])

#train model
xgb_model.fit(X_train, y_train)

print("XGBoost model trained successfully!")

In [ ]:
# Predict class labels
y_pred_xgb = xgb_model.predict(X_test)

# Predict cancellation probabilities
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("Actual Values:     ", y_test[:10].values)
print("Predicted Values:  ", y_pred_xgb[:10])

In [ ]:
# Calculate evaluation metrics

accuracy_xgb = accuracy_score(y_test, y_pred_xgb)

precision_xgb = precision_score(y_test, y_pred_xgb)

recall_xgb = recall_score(y_test, y_pred_xgb)

f1_xgb = f1_score(y_test, y_pred_xgb)

roc_auc_xgb = roc_auc_score(y_test, y_prob_xgb)


print("XGBoost Model Performance")
print("-------------------------")
print(f"Accuracy:  {accuracy_xgb:.4f}")
print(f"Precision: {precision_xgb:.4f}")
print(f"Recall:    {recall_xgb:.4f}")
print(f"F1 Score:  {f1_xgb:.4f}")
print(f"ROC-AUC:   {roc_auc_xgb:.4f}")

In [ ]:
# confusion matrix
cm_xgb = confusion_matrix(y_test, y_pred_xgb)

print("XGBoost Confusion Matrix:")
print(cm_xgb)

In [ ]:
# classification report

print(classification_report(y_test, y_pred_xgb))

### XGBoost – Overall Findings

This section reports XGBoost performance on the same chronological test period. Run the notebook from top to bottom to refresh the results after the updated split and feature set.

## Model Comparison

In this section, we compare the performance of all four classification models:

1. Logistic Regression
2. Decision Tree
3. Random Forest
4. XGBoost

The models will be compared using five evaluation metrics:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

This comparison will help us identify the most suitable model for predicting hotel booking cancellations.

In [ ]:
model_comparison = pd.DataFrame({'Model':['Logistic Regression', 'Decision Tree','Random Forest','XGBoost'],
                                 'Accuracy':[accuracy, accuracy_dt,accuracy_rf,accuracy_xgb],
                                 'Precision':[precision,precision_dt,precision_rf,precision_xgb],
                                 'Recall':[recall,recall_dt,recall_rf,recall_xgb],
                                 'F1 Score':[f1, f1_dt,f1_rf,f1_xgb],
                                 'ROC AUC':[roc_auc,roc_auc_dt,roc_auc_rf,roc_auc_xgb]})

model_comparison = model_comparison.sort_values(by="ROC AUC",ascending=False).reset_index(drop=True)

model_comparison

### Model Comparison – Overall Findings

Compare all four models using metrics calculated on the same chronological test period. The previous random-split results have been cleared because they do not represent the updated evaluation approach. Run the notebook from top to bottom to generate current results.

## Random Forest – Threshold Tuning

The Random Forest threshold is selected using the separate chronological validation dataset. The final test dataset remains untouched until final evaluation.

The validation data was created in the chronological split above; it is not randomly sampled from the training data.

In [ ]:
# The chronological validation set was created in the train-test split cell.
# Keep it separate for threshold selection; do not use it to fit the final test evaluation.
print("Training Data Shape:", X_train.shape)
print("Validation Data Shape:", X_val.shape)
print("Testing Data Shape:", X_test.shape)

In [ ]:
# Create Random Forest pipeline

rf_tuning_model = Pipeline(steps=[('preprocessor', preprocessor),
                                  ('classifier', RandomForestClassifier(n_estimators=100,
                                                                        random_state=49,
                                                                        n_jobs=-1))])

In [ ]:

rf_tuning_model.fit(X_train_new, y_train_new)

In [ ]:
# Generate cancellation probabilities

y_val_prob = rf_tuning_model.predict_proba(X_val)[:, 1]

# Display first 10 probabilities

print("First 10 Cancellation Probabilities:")
print(y_val_prob[:10])

In [ ]:
# Define classification thresholds

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

threshold_results = []

for threshold in thresholds:

    # Convert probabilities into predictions
    y_val_pred = (y_val_prob >= threshold).astype(int)

    # Calculate evaluation metrics
    precision = precision_score(y_val, y_val_pred)
    recall = recall_score(y_val, y_val_pred)
    f1 = f1_score(y_val, y_val_pred)

    threshold_results.append({'Threshold': threshold,'Precision': precision,
                              'Recall': recall,'F1 Score': f1})

# Create comparison DataFrame

threshold_df = pd.DataFrame(threshold_results)

threshold_df

### Threshold Tuning – Overall Findings

The Random Forest model is evaluated at thresholds from 0.30 to 0.60 using the chronological validation set. The latest run selected **0.30** with an F1-score of approximately **65.62%**. Re-run the notebook to regenerate the complete threshold comparison.

### Final Random Forest Model

After selecting the classification threshold using the validation dataset, retrain the pipeline on the combined training and validation data. The chronological test dataset remains untouched and is used only for final evaluation.

In [ ]:
# Create the final Random Forest pipeline
rf_final_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        random_state=49,
        n_jobs=-1
    ))
])

# After selecting the threshold using validation data, refit on train + validation.
X_train_full = pd.concat([X_train, X_val], ignore_index=True)
y_train_full = pd.concat([y_train, y_val], ignore_index=True)

rf_final_model.fit(X_train_full, y_train_full)

print("Final Random Forest Model trained successfully!")

In [ ]:
# Generate cancellation probabilities

y_test_prob_rf = rf_final_model.predict_proba(X_test)[:, 1]

# Apply the selected threshold

best_threshold = 0.40

y_test_pred_rf_tuned = (y_test_prob_rf >= best_threshold).astype(int)

print("First 10 Cancellation Probabilities:")
print(y_test_prob_rf[:10])

print("\nFirst 10 Predictions:")
print(y_test_pred_rf_tuned[:10])

### Final Model Evaluation

We will evaluate the final Random Forest model on the unseen test dataset using the following metrics:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

These metrics will help us understand how effectively the model identifies hotel booking cancellations.

In [ ]:
# Calculate evaluation metrics

rf_tuned_accuracy = accuracy_score(y_test, y_test_pred_rf_tuned)
rf_tuned_precision = precision_score(y_test, y_test_pred_rf_tuned)
rf_tuned_recall = recall_score(y_test, y_test_pred_rf_tuned)
rf_tuned_f1 = f1_score(y_test, y_test_pred_rf_tuned)
rf_tuned_roc_auc = roc_auc_score(y_test, y_test_prob_rf)

# Display results

print("Random Forest - Tuned Threshold (0.40)")
print("---------------------------------------")
print(f"Accuracy:  {rf_tuned_accuracy:.4f}")
print(f"Precision: {rf_tuned_precision:.4f}")
print(f"Recall:    {rf_tuned_recall:.4f}")
print(f"F1 Score:  {rf_tuned_f1:.4f}")
print(f"ROC-AUC:   {rf_tuned_roc_auc:.4f}")

### Final Model Evaluation – Findings

The metrics below are calculated on the latest chronological test period. They should be interpreted as a historical time-based evaluation, not a guarantee of performance on 2026 bookings. The dataset contains historical records from 2015–2017 only.

In [ ]:
# Generate confusion matrix

cm_rf_tuned = confusion_matrix(y_test, y_test_pred_rf_tuned)

print("Confusion Matrix:")
print(cm_rf_tuned)

## Final Model Selection

Random Forest is retained for further development based on the model comparison. The classification threshold is selected on the chronological validation set, and the latest chronological test set is kept separate for final evaluation.

The latest run selected a threshold of 0.30. The final model is retrained using the combined training and validation data before being saved. Historical performance on 2015–2017 records does not guarantee accuracy on 2026 or later bookings.

In [ ]:
import os
import joblib

# Create models folder
os.makedirs("../models", exist_ok=True)

# Save model and selected threshold
model_artifact = {
    "model": rf_final_model,
    "threshold": best_threshold,
    "feature_columns": X.columns.tolist(),
    "model_version": "HotelIQ-RF-chronological-v2",
    "training_data_note": "Historical Hotel Booking Demand data from 2015–2017."
}

joblib.dump(
    model_artifact,
    "../models/hotel_cancellation_model.pkl",
    compress=3
)

print("Model saved successfully!")